In [2]:
#seq2seq英译法案例
#案例步骤：
#1.数据预处理：读取数据，构建词汇表，转换为索引
#2.模型定义：定义编码器和解码器
#3.训练模型：定义损失函数和优化器，训练模型
#4.评估模型：测试模型性能，进行翻译

import re #正则表达式模块
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import time

import random #随机数生成器
import matplotlib.pyplot as plt

device=torch.device("cuda" if torch.cuda.is_available() else "cpu") #设备适配
SOS_token=0 #设置开始标记和结束标记
EOS_token=1

max_length=10 #设置最大句子长度
path='./data/eng-fra-v2.txt'




In [3]:
#数据预处理
#读取文档数据，构建词汇表和索引映射
eng_sentences=[] #英语句子列表
fra_sentences=[]
with open(path,'r',encoding='utf-8') as f:
    sentences=f.read().strip().split('\n')
    print(sentences[:5]) #打印前5行数据
    for sentence in sentences:
        eng,fra=sentence.split('\t')
        eng_sentences.append(eng)
        fra_sentences.append(fra)
    print(eng_sentences[:5])
    print(len(fra_sentences))

#构建两个句子列表的字典对应
eng_fra_dict={}
for i in range(len(eng_sentences)):
    eng_fra_dict.update({eng_sentences[i]:fra_sentences[i]})
# print(eng_fra_dict)

#构建词汇表和索引映射
eng_words=[]
fra_words=[]
for sentence in eng_sentences:
    eng_words.extend(sentence.split(' '))
print(eng_words[:5])

for sentence in fra_sentences:
    fra_words.extend(sentence.split(' '))
print(fra_words[:5])

eng_vocab=list(set(eng_words)) #英语词汇表
fra_vocab=list(set(fra_words))
print(len(eng_vocab))
print(len(fra_vocab))

eng_word_idx={'SOS':SOS_token,'EOS':EOS_token} #英语词汇索引映射
fra_word_idx={'SOS':SOS_token,'EOS':EOS_token} #法语词汇
i=2
j=2
for word in eng_vocab:
    eng_word_idx[word]=i
    i+=1
for word in fra_vocab:
    fra_word_idx[word]=j
    j+=1
# print(fra_word_idx)
# print(fra_word_idx.items())

eng_idx_word={i:w for w,i in eng_word_idx.items()}
fra_idx_word={i:w for w,i in fra_word_idx.items()}
print(eng_idx_word)

eng_seq_idx=[]
fra_seq_idx=[]
#将句子转换为索引序列
for sentence in eng_sentences:
    tem=[eng_word_idx[w] for w in sentence.split(' ')]
    eng_seq_idx.append(tem)
print(eng_seq_idx[:5])

for sentence in fra_sentences:
    tem=[fra_word_idx[w] for w in sentence.split(' ')]
    fra_seq_idx.append(tem)
print(fra_seq_idx[:5])

['i m .\tj ai ans .', 'i m ok .\tje vais bien .', 'i m ok .\tca va .', 'i m fat .\tje suis gras .', 'i m fat .\tje suis gros .']
['i m .', 'i m ok .', 'i m ok .', 'i m fat .', 'i m fat .']
63594
['i', 'm', '.', 'i', 'm']
['j', 'ai', 'ans', '.', 'je']
2801
4343
{0: 'SOS', 1: 'EOS', 2: 'behaving', 3: 'basque', 4: 'blast', 5: 'witch', 6: 'connected', 7: 'freezing', 8: 'extremely', 9: 'conduct', 10: 'violinist', 11: 'responsible', 12: 'bathtub', 13: 'easily', 14: 'wraps', 15: 'contributions', 16: 'threat', 17: 'went', 18: 'scared', 19: 'assertive', 20: 'surgeon', 21: 'tennis', 22: 'gone', 23: 'ruthless', 24: 'useless', 25: 'supposed', 26: 'us', 27: 'present', 28: 'abroad', 29: 'wonderful', 30: 'nice', 31: 'curt', 32: 'miss', 33: 'actor', 34: 'nonsmoker', 35: 'west', 36: 'after', 37: 'australians', 38: 'art', 39: 'plastered', 40: 'happy', 41: 'adults', 42: 'forty', 43: 'breaking', 44: 'herself', 45: 'everything', 46: 'encouraged', 47: 'weeks', 48: 'shower', 49: 'coward', 50: 'corner', 51: '

In [ ]:
#构建数据集
eng_fra_list=list(eng_fra_dict.items())

#定义数据集类
class SeqDataset(Dataset):
    def __init__(self,eng_fra_list):
        self.eng_fra_pairs=eng_fra_list
        self.length=len(eng_fra_list)
    def __len__(self):
        return self.length
    def __getitem__(self,idx):
        eng=self.eng_fra_pairs[idx][0]
        fra=self.eng_fra_pairs[idx][1]
        eng_idx=[eng_word_idx[w] for w in eng.split(' ')]
        fra_idx=[fra_word_idx[w] for w in fra.split(' ')]
        #添加开始标记
        fra_idx.insert(0,fra_word_idx['SOS'])
        eng_idx.append(eng_word_idx['EOS'])
        fra_idx.append(fra_word_idx['EOS']) #添加结束标记

        #源序列只有结束标记，目标序列既有开始标记也有结束标记
        x=torch.tensor(eng_idx)
        y=torch.tensor(fra_idx)
        return x,y

#实例化数据集和数据加载器
dataset=SeqDataset(eng_fra_list)
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)
for x,y in dataloader:
    print(x)
    print(y)


tensor([[ 480, 1758,  727, 1682,    1]])
tensor([[   0, 2193,  621,  346, 2478,    1]])
tensor([[1876,  517,  809, 2622,   81, 1682,    1]])
tensor([[   0, 1755, 3874, 2373, 1974, 1206, 2478,    1]])
tensor([[1355, 1758,  435, 1791, 1682,    1]])
tensor([[   0,  890, 3457,  866, 1316, 2478,    1]])
tensor([[1806, 2527, 1674,  922,  746,  809,  583, 1253, 1682,    1]])
tensor([[   0,   81, 1064, 1552, 2138, 2529, 3908, 1039, 2478,    1]])
tensor([[1271,  517, 1725,  123,  922, 2395, 1682,    1]])
tensor([[   0, 1428, 1901, 3874, 2925, 4104, 1201,  652, 3368, 2478,    1]])
tensor([[1806, 2527,  649, 1806,  645,  642,   44, 2757, 1682,    1]])
tensor([[   0, 1079, 3248, 3675, 2214, 3659,  111, 1861,  923, 2478,    1]])
tensor([[1806, 2692,  809,  361, 2500, 1682,    1]])
tensor([[   0, 1079, 3248, 2529,  135, 2675, 2478,    1]])
tensor([[1806, 2527, 2000, 2724, 1051, 1032, 1682,    1]])
tensor([[   0, 1079,  829, 2424,  302, 1586, 2478,    1]])
tensor([[ 480, 1758, 1029, 1682,    1]])
ten

In [ ]:
#定义编码器和解码器
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(eng_word_idx),128) #词嵌入层
        self.gru=nn.GRU(128,256,1,batch_first=True) #GRU层，隐藏状态维度为256
    def forward(self,x,h0):
        x=self.embed(x)
        output,hn=self.gru(x,h0)
        return output,hn
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256,device=device)

#检验编码器
encoder=Encoder().to(device)
for x,y in dataloader:
    x=x.to(device)
    y=y.to(device)
    h0=encoder.init_hidden(x.size(0))
    output,hn=encoder(x,h0)
    print(output.shape)
    print(hn.shape)


#这里output的shape是(batch_size,seq_len,hidden_size)，hn的shape是(num_layers*num_directions,batch_size,hidden_size)


In [ ]:
#定义解码器
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(fra_word_idx),128)
        self.gru=nn.GRU(128,256,1,batch_first=True)
        self.fc1=nn.Linear(256,len(fra_word_idx))
    def forward(self,x,h0):
        x=self.embed(x)
        x=F.relu(x) #relu激活函数，防止梯度消失，过拟合
        output,hn=self.gru(x,h0)
        output=self.fc1(output)
        return output,hn
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256,device=device)
#检验解码器
decoder=Decoder().to(device)
for x,y in dataloader:
    x=x.to(device)
    y=y.to(device)
    h0=decoder.init_hidden(x.size(0))
    output,hn=decoder(x,h0)
    # print(output.shape)
    # print(hn.shape)
    print(y.shape)



In [ ]:
#添加attention机制
#Q:解码器当前隐藏状态
#K:编码器输出的所有隐藏状态 这里是output
#V和K相同
#上下文和输入拼接后输入解码器
class AttentionDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(fra_word_idx),128) #词嵌入层，输入维度是目标词汇表大小，输出维度是128，为词向量维数
        self.gru=nn.GRU(128+256,256,1,batch_first=True) #输入维度是输入维度加上上下文维度
        self.fc1=nn.Linear(256,len(fra_word_idx))
        self.fc2=nn.Linear(256,256)

    def forward(self,x,h0,encoder_outputs):
        x=self.embed(x)
        x=nn.Dropout(0.3)(x) #添加dropout层，防止过拟合
        #Q:h0
        #K:encoder_outputs
        #V:encoder_outputs
        encoder_outputs=self.fc2(encoder_outputs) #将编码器输出映射到解码器隐藏状态维度
        weights=torch.bmm(h0,encoder_outputs.transpose(2,1))/torch.sqrt(torch.tensor(256)) #计算注意力权重
        weights=F.softmax(weights,dim=1) #将注意力权重归一化，确保每个时间步的注意力权重和为1
        context=torch.bmm(weights,encoder_outputs) #计算上下文向量

        context_expanded=context.expand(-1,x.size(1),-1) #将上下文向量扩展到与输入相同的时间步长，为了与输入拼接
        decoder_input=torch.cat((context_expanded,x),dim=2) #将上下文向量和输入拼接
        output,hn=self.gru(decoder_input,h0)
        output=self.fc1(output)
        return output,hn
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256,device=device)


In [ ]:


#验证解码器
attention_decoder=AttentionDecoder().to(device)
for x,y in dataloader:
    x=x.to(device)
    y=y.to(device)
    h0_decoder=attention_decoder.init_hidden(x.size(0))
    h0_encoder=encoder.init_hidden(x.size(0))
    encoder_output,hn=encoder(x,h0_encoder)
    output,hn=attention_decoder(x,h0_decoder,encoder_output)
    print(output.shape)
    print(hn.shape)

In [6]:
#实现加性注意力机制
class AdditiveAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(fra_word_idx),128)
        self.gru=nn.GRU(128+256,256,1,batch_first=True)
        self.fc1=nn.Linear(256,256)
        self.fc2=nn.Linear(256,256)
        self.fc3=nn.Linear(256,1)
        self.fc4=nn.Linear(256,len(fra_word_idx))
    def forward(self,x,h0_decoder,encoder_outputs):
        x=self.embed(x)
        q=self.fc1(h0_decoder)
        k=self.fc2(encoder_outputs)
        scores=self.fc3(torch.tanh(q+k))
        weights=F.softmax(scores,dim=1)
        context=torch.bmm(weights.transpose(1,2),encoder_outputs)
        context_expanded=context.expand(-1,x.size(1),-1)
        decoder_input=torch.cat((context_expanded,x),dim=2)
        decoder_output,hn_decoder=self.gru(decoder_input,h0_decoder)
        outputs=self.fc4(decoder_output)
        return outputs,hn_decoder

        
       

In [ ]:
#训练模型
#教师强制训练
encoder=Encoder().to(device)
decoder=AttentionDecoder().to(device)
criterion=nn.CrossEntropyLoss()
epochs=1
encoder_optimizer=optim.Adam(encoder.parameters(),lr=0.001)
decoder_optimizer=optim.Adam(decoder.parameters(),lr=0.001)
for epoch in range(epochs):
    encoder.train()
    decoder.train()
    total_samples,total_correct,total_loss,start=0,0,0,time.time()
    for x,y in dataloader:
        x=x.to(device)
        y=y.to(device)
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()
        h0_encoder=encoder.init_hidden(x.size(0))
        h0_decoder=decoder.init_hidden(x.size(0))
        encoder_output,hn_encoder=encoder(x,h0_encoder)
        output,hn_decoder=decoder(y[:,:-1],h0_decoder,encoder_output) #教师强制训练，输入是目标序列的前n-1个时间步，输出是目标序列的后n个时间步
        loss=criterion(output.permute(0,2,1),y[:,1:]) #计算损失值，将输出的维度从(batch_size,seq_len,hidden_size)转换为(batch_size,hidden_size,seq_len)，因为criterion函数需要输入的维度是(batch_size,hidden_size,seq_len)，然后与目标序列的后n个时间步进行对比
        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()
        total_loss+=loss.item()*x.size(0)
        total_samples+=x.size(1)*x.size(0) #计算总样本数，x.size(1)是序列长度，因为每个时间步都要计算损失值，所以总样本数是序列长度乘以批量大小
        total_correct+=(torch.argmax(output,dim=2)==y[:,1:]).sum() #计算准确率，将输出的维度从(batch_size,seq_len,hidden_size)转换为(batch_size,seq_len)，因为argmax函数需要输入的维度是(batch_size,seq_len)，然后与目标序列的后n个时间步进行对比
    end=time.time()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/total_samples:.4f}, Accuracy: {total_correct/total_samples:.4f}, Time: {end-start:.2f}s')







KeyboardInterrupt: 

In [ ]:
#在阿里云平台上训练模型，使用GPU加速，训练时间大大缩短，模型性能得到提升。训练完成后，可以保存模型参数，以便后续使用和部署。模型训练准确率为67%,发现使用缩放点积注意力时模型性能较差，故选择改用加性注意力机制。模型准确率上升至97%

In [ ]:
# 实现模型评估
encoder=Encoder().to(device)
encoder.load_state_dict(torch.load('./models/encoder.pth')) #加载模型参数

decoder=AttentionDecoder().to(device)
decoder.load_state_dict(torch.load('./models/decoder.pth')) #加载模型参数

encoder.eval()
decoder.eval() #切换为评估模式

pairs = [
    ['i m at the airport right now .', 'je me trouve actuellement a l aeroport .'],
    ['i m aware of the possibility .', 'je suis conscient de la possibilite .']
]

with torch.no_grad(): #关闭梯度计算
    for eng_sentence,fra_sentence in pairs:
        eng_idx=[eng_word_idx[w] for w in eng_sentence.split(' ')]
        eng_idx.append(eng_word_idx['EOS']) #句尾添加结束标记

        x=torch.tensor(eng_idx,device=device)
        fra_pred=[]
        max_len=10 #设置最大生成长度

        h0_encoder=encoder.init_hidden(1) #batch_size=1
        encoder_outputs,hn_encoder=encoder(x,h0_encoder) #将数据送入编码器编码，

        h0_decoder=hn_encoder #解码器的初始状态设置为编码器的最后一个隐藏状态
        decoder_input=torch.tensor([[eng_word_idx['SOS']]],device=device) #解码器的初始输入为开始标记
        
        for i in range(max_len): #限制最大长度
            decoder_outputs,h0_decoder=decoder(decoder_input,h0_decoder,encoder_outputs) #将当前输入、隐藏状态和编码器输出送入解码器，得到当前时间步的输出和新的隐藏状态，新的隐藏状态将用于下一个时间步的解码
            topv,topi=decoder_outputs.topk(1) #获取当前时间步输出的最大值和对应的索引，topi是一个张量，包含了当前时间步预测的词汇索引
            

            if topi.item()==fra_word_idx['EOS']: #等于结束标记时停止输出
                break

            fra_pred.append(fra_idx_word[topi.item()]) #将预测的索引转换为对应的词汇，并添加到预测结果列表中
            decoder_input=topi.squeeze().detach().unsqueeze(0).unsqueeze(0) #将预测的索引作为下一个时间步的输入，进行维度调整以适应解码器的输入要求
        fra_tokens=fra_sentence.split(' ')

        print(f'eng:{eng_sentence}')
        print(f'fra:{fra_sentence}')
        print(f'fra_pred:{fra_pred}')




[tensor([ 286, 1922, 1189, 2054,  275, 1234,  688,   55,    1]), tensor([ 286, 1922, 1448, 2358, 2054, 2499,   55,    1])]
[tensor([   0, 4001, 1794, 1161, 1281, 2073, 3148,  200,   99,    1]), tensor([   0, 4001, 3699, 3828, 4181,  982, 3466,   99,    1])]


In [ ]:
# 确保先运行了定义Encoder和AttentionDecoder类的单元格
# 确保运行了词汇表构建的代码（eng_word_idx, fra_word_idx, fra_idx_word等）

# 加载训练好的模型
encoder = Encoder().to(device)
encoder.load_state_dict(torch.load('./models/encoder.pth'))

decoder = AttentionDecoder().to(device)
decoder.load_state_dict(torch.load('./models/decoder.pth'))

# 你的评估数据
pairs = [
    ['i m at the airport right now .', 'je me trouve actuellement a l aeroport .'],
    ['i m aware of the possibility .', 'je suis conscient de la possibilite .']
]

encoder.eval()
decoder.eval()

# 用于计算准确率
total_correct = 0
total_tokens = 0
with torch.no_grad():
    for eng_sentence, fra_sentence in pairs:
        # 处理英语输入序列
        eng_tokens = eng_sentence.split(' ')
        eng_idx = [eng_word_idx[w] for w in eng_tokens]
        eng_idx.append(eng_word_idx['EOS'])
        input_tensor = torch.tensor([eng_idx], device=device)  # [1, seq_len]
        
        # 编码器处理
        h0_encoder = encoder.init_hidden(1)  # batch_size=1
        encoder_outputs, hn_encoder = encoder(input_tensor, h0_encoder)
        
        # 解码器初始化 - 从SOS开始
        decoder_input = torch.tensor([[fra_word_idx['SOS']]], device=device)  # [1, 1]
        h0_decoder = hn_encoder
        predicted_words = []
        
        # 生成法语翻译
        max_length = 10  # 最大生成长度
        for i in range(max_length):
            output, h0_decoder = decoder(decoder_input, h0_decoder, encoder_outputs)
            topv, topi = output.topk(1)
            predicted_idx = topi.item()
            
            if predicted_idx == fra_word_idx['EOS']:
                break
            
            predicted_words.append(fra_idx_word[predicted_idx])
            decoder_input = topi.squeeze().detach().unsqueeze(0).unsqueeze(0)
        
        # 处理真实法语序列（去掉EOS）
        fra_tokens = fra_sentence.split(' ')
        
        # 打印结果
        print(f"英语: {eng_sentence}")
        print(f"法语(真实): {fra_sentence}")
        print(f"法语(预测): {' '.join(predicted_words)}")
        
#         # 计算准确率（可选）
#         min_length = min(len(fra_tokens), len(predicted_words))
#         for i in range(min_length):
#             if fra_tokens[i] == predicted_words[i]:
#                 total_correct += 1
#         total_tokens += len(fra_tokens)

# # 打印准确率
# if total_tokens > 0:
#     accuracy = total_correct / total_tokens
#     print(f"评估完成！准确率: {accuracy:.4f}")
# else:
#     print("评估完成！")